In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install wandb --quiet

## Config

Set `REPO_ROOT` to the repo root on your Google Drive. All other paths are relative to it.

In [ ]:
import os
import sys
import json
import pickle as pkl
from dataclasses import asdict
from typing import Optional, Tuple

import torch
import wandb
from torch.utils.data import DataLoader

In [ ]:
REPO_ROOT = "/content/drive/MyDrive/Rutishauser_Lab/AI4Sci/METHODS/DiffusionBANMap/"
CONFIG_FILE = "config/diffusion_full_cfd.json"
RUN_NAME = "run3"
RUN_NOTES = "Try Min-SNR-gamma weighting"
RESUME_CKPT_PATH = None
NUM_WORKERS = 0  # keep 0; data lives in RAM after first load

## Imports

In [ ]:
sys.path.insert(0, REPO_ROOT)

import importlib.metadata
if not hasattr(importlib.metadata, '_patched'):

    _original_version = importlib.metadata.version
    def _patched_version(name):
        try:
            return _original_version(name)
        except importlib.metadata.PackageNotFoundError:
            return 'unknown'
    importlib.metadata.version = _patched_version
    importlib.metadata._patched = True

from data_utils import PreloadedDataset
from models.beta_vae import BetaVAE
from models.unet import UNet
from scripts.config import BetaVAEConfig, DiffusionConfig
from scripts.diffusion_trainer import DiffusionTrainer
from utils import discover_device, count_model_params

## Load config & build UNet

In [ ]:
config_path = os.path.join(REPO_ROOT, CONFIG_FILE)
with open(config_path) as f:
    raw = json.load(f)

# Resolve relative paths to absolute Drive paths
for key in ("train_dataset_path", "test_dataset_path", "output_dir",
            "betavae_config_path", "betavae_ckpt_path"):
    if key in raw and raw[key]:
        raw[key] = os.path.join(REPO_ROOT, raw[key].lstrip("./"))

config = DiffusionConfig(**raw)

device = torch.device(discover_device())
print(f"Device: {device}")

model = UNet(
    in_channels=config.in_channels,
    image_size=config.image_size,
    model_channels=config.model_channels,
    channel_mult=config.channel_mult,
    num_res_blocks=config.num_res_blocks,
    attention_resolutions=config.attention_resolutions,
    dropout=config.dropout,
    latent_dim=config.latent_dim,
)
count_model_params(model)

## Load frozen β-VAE

In [ ]:
with open(config.betavae_config_path) as f:
    vae_cfg = BetaVAEConfig(**json.load(f))

vae = BetaVAE(
    input_channels=vae_cfg.input_channels,
    input_height=vae_cfg.input_height,
    input_width=vae_cfg.input_width,
    latent_dim=vae_cfg.latent_dim,
    hidden_dim=vae_cfg.hidden_dim,
    beta=vae_cfg.beta,
    device=device,
)
ckpt = torch.load(config.betavae_ckpt_path, map_location=device)
vae.load_state_dict(ckpt["model_state_dict"])
vae.eval()
print(f"Loaded β-VAE from step {ckpt['step']}")

## Data

In [ ]:
def load_dataloader(path: Optional[str], shuffle: bool) -> Optional[DataLoader]:
    if not path:
        return None
    with open(path, "rb") as f:
        tensor = pkl.load(f)
    tensor = tensor.to(device)
    dataset = PreloadedDataset(tensor)
    loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=shuffle, num_workers=0)
    print(f"{os.path.basename(path)}: {len(dataset)} images, {len(loader)} batches")
    return loader

train_dataloader = load_dataloader(config.train_dataset_path, shuffle=True)
val_dataloader = load_dataloader(config.test_dataset_path,  shuffle=False)

## Wandb

In [ ]:
wandb.login(key="...")

wandb.init(
    project="diffusion_ban_map",
    name=f"{config.experiment_name}_{RUN_NAME}",
    notes=RUN_NOTES,
    config=asdict(config),
)

## Train

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)

trainer = DiffusionTrainer(
    model=model,
    vae_model=vae,
    optimizer=optimizer,
    scheduler=None,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    config=config,
    device=device,
)

trainer.train(resume_ckpt_path=RESUME_CKPT_PATH)